# Hyperparameter Optimization
This notebook performs hyperparameter optimization for multiple shallow learning models.

The following classifiers are optimized:

- Logistic Regression
- Support Vector Machine (SVM)
- k-Nearest Neighbors (kNN)
- Random Forest

Grid Search and cross-validation are used to identify the best hyperparameter combinations.

# Imports

In [1]:
# ============================================================
# IMPORTS
# ============================================================

# Data manipulation
import pandas as pd
import numpy as np

# Ignore warnings
import warnings
warnings.filterwarnings("ignore")

# Scaling
from sklearn.preprocessing import StandardScaler

# Models
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier

# Hyperparameter optimization
from sklearn.model_selection import GridSearchCV

# Metrics
from sklearn.metrics import (

    accuracy_score,
    recall_score,
    f1_score,
    confusion_matrix,
    matthews_corrcoef
)

# Visualization
import matplotlib.pyplot as plt


# Load Reduced Dataset
The reduced dataset generated using Sequential Forward Selection (SFS) is used for training.

In [2]:
# ============================================================
# LOAD REDUCED TRAIN DATASET
# ============================================================

# Reduced SFS dataset

X_train = pd.read_csv(
    "sfs_dataset.csv"
)

# Original training dataset
# used only to recover labels

train_df = pd.read_csv(
    "00-train.csv"
)

# Labels

y_train = train_df["Class"]

print("="*60)
print("TRAIN DATASET")
print("="*60)

print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)

TRAIN DATASET
X_train shape: (198, 10)
y_train shape: (198,)


# Load Tuning and External Datasets

In [4]:
# ============================================================
# LOAD OTHER DATASETS
# ============================================================

# Load datasets

tuning_df = pd.read_csv(
    "01-tuning.csv"
)

external_df = pd.read_csv(
    "02-external.csv"
)

# Keep SAME selected variables

X_tuning = tuning_df[
    X_train.columns
]

X_external = external_df[
    X_train.columns
]

# Labels

y_tuning = tuning_df["Class"]

y_external = external_df["Class"]

print("="*60)
print("OTHER DATASETS")
print("="*60)

print("\nX_tuning shape:")
print(X_tuning.shape)

print("\nX_external shape:")
print(X_external.shape)

OTHER DATASETS

X_tuning shape:
(62, 10)

X_external shape:
(11182, 10)


# Feature Scaling
Scaling is required for:

- Logistic Regression
- SVM
- kNN


In [5]:
# ============================================================
# FEATURE SCALING
# ============================================================

# Create scaler

scaler = StandardScaler()

# Learn parameters from training data

scaler.fit(X_train)

# Transform datasets

X_train_scaled = scaler.transform(
    X_train
)

X_tuning_scaled = scaler.transform(
    X_tuning
)

X_external_scaled = scaler.transform(
    X_external
)

print("Scaling completed successfully.")

Scaling completed successfully.


# Evaluation Function

In [6]:
# ============================================================
# EVALUATION FUNCTION
# ============================================================

def evaluate_model(

    y_true,
    y_pred
):

    # Confusion matrix

    cm = confusion_matrix(
        y_true,
        y_pred
    )

    # Extract values

    tn, fp, fn, tp = cm.ravel()

    # Accuracy

    acc = accuracy_score(
        y_true,
        y_pred
    )

    # Sensitivity

    sen = recall_score(
        y_true,
        y_pred,
        pos_label="APP"
    )

    # Specificity

    spe = tn / (tn + fp)

    # MCC

    mcc = matthews_corrcoef(
        y_true,
        y_pred
    )

    # Weighted F1-score

    f1 = f1_score(
        y_true,
        y_pred,
        average="weighted"
    )

    return {

        "ACC": acc,
        "SEN": sen,
        "SPE": spe,
        "MCC": mcc,
        "F1": f1,
        "CM": cm
    }

# Logistic Regression — Grid Search

In [7]:
# ============================================================
# LOGISTIC REGRESSION — GRID SEARCH
# ============================================================

# Hyperparameter grid

param_grid_lr = {

    "C": [0.01, 0.1, 1, 10],

    "solver": [

        "liblinear",
        "lbfgs"
    ],

    "penalty": [

        "l2"
    ]
}

# Base model

lr_base = LogisticRegression(
    random_state=42,
    max_iter=1000
)

# Grid Search

grid_lr = GridSearchCV(

    estimator=lr_base,

    param_grid=param_grid_lr,

    scoring="f1_weighted",

    cv=5,

    n_jobs=-1
)

# Train all combinations

grid_lr.fit(

    X_train_scaled,
    y_train
)

# Best model

best_lr = grid_lr.best_estimator_

# Predictions

y_pred_lr = best_lr.predict(
    X_tuning_scaled
)

# Evaluation

results_lr = evaluate_model(

    y_tuning,
    y_pred_lr
)

print("="*60)
print("OPTIMIZED LOGISTIC REGRESSION")
print("="*60)

print("\nBest hyperparameters:")
print(grid_lr.best_params_)

print("\nBest CV score:")
print(grid_lr.best_score_)

print("\nEvaluation metrics:")
print(results_lr)

OPTIMIZED LOGISTIC REGRESSION

Best hyperparameters:
{'C': 0.01, 'penalty': 'l2', 'solver': 'liblinear'}

Best CV score:
0.8099486168886161

Evaluation metrics:
{'ACC': 0.7258064516129032, 'SEN': 0.7096774193548387, 'SPE': np.float64(0.7096774193548387), 'MCC': 0.45184805705753195, 'F1': 0.7257351027842831, 'CM': array([[22,  9],
       [ 8, 23]])}


# SVM — Grid Search

In [8]:
# ============================================================
# SVM — GRID SEARCH
# ============================================================

# Hyperparameter grid

param_grid_svm = {

    "C": [0.1, 1, 10],

    "kernel": [

        "linear",
        "rbf"
    ],

    "gamma": [

        "scale",
        "auto"
    ]
}

# Base model

svm_base = SVC(
    random_state=42
)

# Grid Search

grid_svm = GridSearchCV(

    estimator=svm_base,

    param_grid=param_grid_svm,

    scoring="f1_weighted",

    cv=5,

    n_jobs=-1
)

# Train

grid_svm.fit(

    X_train_scaled,
    y_train
)

# Best model

best_svm = grid_svm.best_estimator_

# Predictions

y_pred_svm = best_svm.predict(
    X_tuning_scaled
)

# Evaluation

results_svm = evaluate_model(

    y_tuning,
    y_pred_svm
)

print("="*60)
print("OPTIMIZED SVM")
print("="*60)

print("\nBest hyperparameters:")
print(grid_svm.best_params_)

print("\nBest CV score:")
print(grid_svm.best_score_)

print("\nEvaluation metrics:")
print(results_svm)

OPTIMIZED SVM

Best hyperparameters:
{'C': 1, 'gamma': 'scale', 'kernel': 'rbf'}

Best CV score:
0.8310087776663636

Evaluation metrics:
{'ACC': 0.7741935483870968, 'SEN': 0.7419354838709677, 'SPE': np.float64(0.7419354838709677), 'MCC': 0.5495319562599505, 'F1': 0.7739583333333333, 'CM': array([[23,  8],
       [ 6, 25]])}


# kNN — Grid Search

In [10]:
# ============================================================
# KNN — GRID SEARCH
# ============================================================

# Hyperparameter grid

param_grid_knn = {

    "n_neighbors": [

        3,
        5,
        7,
        9
    ],

    "weights": [

        "uniform",
        "distance"
    ]
}

# Base model

knn_base = KNeighborsClassifier()

# Grid Search

grid_knn = GridSearchCV(

    estimator=knn_base,

    param_grid=param_grid_knn,

    scoring="f1_weighted",

    cv=5,

    n_jobs=-1
)

# Train

grid_knn.fit(

    X_train_scaled,
    y_train
)

# Best model

best_knn = grid_knn.best_estimator_

# Predictions

y_pred_knn = best_knn.predict(
    X_tuning_scaled
)

# Evaluation

results_knn = evaluate_model(

    y_tuning,
    y_pred_knn
)

print("="*60)
print("OPTIMIZED KNN")
print("="*60)

print("\nBest hyperparameters:")
print(grid_knn.best_params_)

print("\nBest CV score:")
print(grid_knn.best_score_)

print("\nEvaluation metrics:")
print(results_knn)

OPTIMIZED KNN

Best hyperparameters:
{'n_neighbors': 3, 'weights': 'uniform'}

Best CV score:
0.7916664745933039

Evaluation metrics:
{'ACC': 0.7741935483870968, 'SEN': 0.8387096774193549, 'SPE': np.float64(0.8387096774193549), 'MCC': 0.5530100413375022, 'F1': 0.7732497387669802, 'CM': array([[26,  5],
       [ 9, 22]])}


# Random Forest — Grid Search

In [11]:
# ============================================================
# RANDOM FOREST — GRID SEARCH
# ============================================================

# Hyperparameter grid

param_grid_rf = {

    "n_estimators": [

        50,
        100,
        200
    ],

    "max_depth": [

        3,
        5,
        10,
        None
    ],

    "min_samples_split": [

        2,
        5
    ]
}

# Base model

rf_base = RandomForestClassifier(
    random_state=42
)

# Grid Search

grid_rf = GridSearchCV(

    estimator=rf_base,

    param_grid=param_grid_rf,

    scoring="f1_weighted",

    cv=5,

    n_jobs=-1
)

# Train

grid_rf.fit(

    X_train,
    y_train
)

# Best model

best_rf = grid_rf.best_estimator_

# Predictions

y_pred_rf = best_rf.predict(
    X_tuning
)

# Evaluation

results_rf = evaluate_model(

    y_tuning,
    y_pred_rf
)

print("="*60)
print("OPTIMIZED RANDOM FOREST")
print("="*60)

print("\nBest hyperparameters:")
print(grid_rf.best_params_)

print("\nBest CV score:")
print(grid_rf.best_score_)

print("\nEvaluation metrics:")
print(results_rf)

OPTIMIZED RANDOM FOREST

Best hyperparameters:
{'max_depth': 10, 'min_samples_split': 2, 'n_estimators': 100}

Best CV score:
0.9444804514174905

Evaluation metrics:
{'ACC': 0.7741935483870968, 'SEN': 0.7741935483870968, 'SPE': np.float64(0.7741935483870968), 'MCC': 0.5483870967741935, 'F1': 0.7741935483870968, 'CM': array([[24,  7],
       [ 7, 24]])}


# Optimized Model Comparison

In [12]:
# ============================================================
# COMPARISON TABLE
# ============================================================

comparison_df = pd.DataFrame({

    "Model": [

        "Logistic Regression",
        "SVM",
        "kNN",
        "Random Forest"
    ],

    "ACC": [

        results_lr["ACC"],
        results_svm["ACC"],
        results_knn["ACC"],
        results_rf["ACC"]
    ],

    "SEN": [

        results_lr["SEN"],
        results_svm["SEN"],
        results_knn["SEN"],
        results_rf["SEN"]
    ],

    "SPE": [

        results_lr["SPE"],
        results_svm["SPE"],
        results_knn["SPE"],
        results_rf["SPE"]
    ],

    "MCC": [

        results_lr["MCC"],
        results_svm["MCC"],
        results_knn["MCC"],
        results_rf["MCC"]
    ],

    "F1": [

        results_lr["F1"],
        results_svm["F1"],
        results_knn["F1"],
        results_rf["F1"]
    ]
})

print("="*60)
print("OPTIMIZED MODEL COMPARISON")
print("="*60)

print(comparison_df)

OPTIMIZED MODEL COMPARISON
                 Model       ACC       SEN       SPE       MCC        F1
0  Logistic Regression  0.725806  0.709677  0.709677  0.451848  0.725735
1                  SVM  0.774194  0.741935  0.741935  0.549532  0.773958
2                  kNN  0.774194  0.838710  0.838710  0.553010  0.773250
3        Random Forest  0.774194  0.774194  0.774194  0.548387  0.774194


# External Evaluation

In [13]:
# ============================================================
# FINAL EXTERNAL EVALUATION
# ============================================================

# Logistic Regression

external_lr = evaluate_model(

    y_external,

    best_lr.predict(
        X_external_scaled
    )
)

# SVM

external_svm = evaluate_model(

    y_external,

    best_svm.predict(
        X_external_scaled
    )
)

# KNN

external_knn = evaluate_model(

    y_external,

    best_knn.predict(
        X_external_scaled
    )
)

# Random Forest

external_rf = evaluate_model(

    y_external,

    best_rf.predict(
        X_external
    )
)

# Comparison table

external_df = pd.DataFrame({

    "Model": [

        "Logistic Regression",
        "SVM",
        "kNN",
        "Random Forest"
    ],

    "ACC": [

        external_lr["ACC"],
        external_svm["ACC"],
        external_knn["ACC"],
        external_rf["ACC"]
    ],

    "MCC": [

        external_lr["MCC"],
        external_svm["MCC"],
        external_knn["MCC"],
        external_rf["MCC"]
    ],

    "F1": [

        external_lr["F1"],
        external_svm["F1"],
        external_knn["F1"],
        external_rf["F1"]
    ]
})

print("="*60)
print("EXTERNAL DATASET RESULTS")
print("="*60)

print(external_df)

EXTERNAL DATASET RESULTS
                 Model       ACC       MCC        F1
0  Logistic Regression  0.776158  0.225453  0.845188
1                  SVM  0.811483  0.236382  0.867900
2                  kNN  0.697728  0.169366  0.791698
3        Random Forest  0.833840  0.278302  0.882336
